# 04. Handoffs (핸드오프 심화)

**핸드오프(Handoff)** 는 에이전트가 특정 작업을 다른 에이전트에게 위임하는 기능입니다.  
핸드오프는 LLM에게 **도구(tool)** 로 표현됩니다.  
예) `Refund Agent`로의 핸드오프 → LLM 도구 이름: `transfer_to_refund_agent`

| 방식 | 예시 | 특징 |
|------|------|------|
| Agent 직접 전달 | `handoffs=[billing_agent]` | 간단, 기본 동작 |
| `handoff()` 함수 | `handoffs=[handoff(billing_agent, on_handoff=...)]` | 콜백, 이름 재정의, 입력 필터 등 고급 옵션 |

**`handoff()` 함수의 주요 파라미터:**

| 파라미터 | 설명 |
|---------|------|
| `agent` | 위임할 대상 에이전트 |
| `tool_name_override` | LLM에게 노출되는 도구 이름 재정의 (기본: `transfer_to_<agent_name>`) |
| `tool_description_override` | 도구 설명 재정의 |
| `on_handoff` | 핸드오프 호출 시 실행되는 콜백 함수 |
| `input_type` | LLM이 핸드오프 시 함께 전달할 데이터 타입 (Pydantic 모델) |
| `input_filter` | 다음 에이전트에게 전달되는 대화 기록 필터링 함수 |
| `is_enabled` | 핸드오프 활성화 여부 (bool 또는 런타임 함수) |

In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
Model = "gpt-5.4-mini"

In [27]:
import os
from dotenv import load_dotenv
#load_dotenv() 

# 현재 실행 중인 Python 파일의 디렉토리를 기준으로 .env 경로 설정
#current_dir = os.path.dirname(os.path.abspath(__file__))
current_dir = os.getcwd() # 현재경로
parent_dir = os.path.dirname(current_dir) # os.path.dirname()을 한 번 감싸주면 상위 폴더 경로가 됩니다.
dotenv_path = os.path.join(parent_dir, '.env') # 3. 상위 폴더에 있는 진짜 .env 파일 경로 지정

# 디버깅을 위한 출력
print("수정된 .env 예상 경로:", dotenv_path)

# 4. override=True 옵션을 주어 기존의 잘못된 키 값을 확실하게 덮어씁니다.
is_loaded = load_dotenv(dotenv_path, override=True)
print("Env loaded:", is_loaded)

# 5. API 키 확인 (앞뒤 글자만 확인)
api_key = os.environ.get("OPENAI_API_KEY")
if api_key:
    print(f"로드된 API Key: {api_key[:12]}...{api_key[-4:]}")
else:
    print("❌ 여전히 API Key를 찾을 수 없습니다. 경로를 다시 확인해주세요.")
    
Model = "gpt-5.4-mini"

수정된 .env 예상 경로: C:\Users\KOSA\Desktop\Agentic_AI_GPT_MCP-main\.env
Env loaded: True
로드된 API Key: sk-proj-OibW...3DAA


In [28]:
# 사용 중인 에이전트 라이브러리에 따라 가져오는 함수명이 다를 수 있습니다.
from agents import set_tracing_export_api_key

# .env에서 읽어온 키를 트레이싱 모듈에 명시적으로 전달
#api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    set_tracing_export_api_key(api_key)   

## 1. handoff() 함수 기본 사용법

Agent 인스턴스를 직접 전달하는 방식과 `handoff()` 함수를 사용하는 방식은 **동일하게 동작**합니다.  
`handoff()` 함수 형태는 추가 옵션이 필요할 때 사용합니다.

In [3]:
from agents import Agent, Runner, handoff

# --- 1. 하위 에이전트(Sub-Agent) 정의 ---

# 청구(Billing) 전담 에이전트
billing_agent = Agent(
    name="Billing_agent",
    instructions="당신은 청구 관련 질문을 전문으로 처리합니다.",
    model=Model
)

# 환불(Refund) 전담 에이전트
refund_agent = Agent(
    name="Refund_agent",
    instructions="당신은 환불 요청을 전문으로 처리합니다.",
    model=Model
)

# --- 2. 트리아지(분류) 에이전트 정의 ---

# 사용자 요청을 분석한 뒤, 적절한 하위 에이전트로 핸드오프(handoff)하는 역할
triage_agent = Agent(
    name="Triage_agent",
    instructions="사용자 요청을 분류하여 적합한 에이전트에게 넘겨주세요.",
    model=Model,
    handoffs=[
        # 방법 1 : Agent 인스턴스를 직접 전달 → 자동으로 tool_name/description 생성
        billing_agent,

        # 방법 2 : handoff() 함수로 감싸서 전달 → tool 이름·설명을 직접 커스터마이징
        handoff(
            refund_agent,
            tool_name_override="request_refund",           # LLM이 호출할 tool 이름 재정의
            tool_description_override="환불 요청이 있을 때 사용",  # tool 설명 재정의
        ),
    ],
)

# --- 3. 실행 ---
# triage_agent가 입력을 받아 내부적으로 billing_agent 또는 refund_agent로 핸드오프
result = await Runner.run(triage_agent, input="제 청구서에 오류가 있는 것 같습니다.")

# 최종 출력 — 핸드오프된 에이전트(billing_agent)가 생성한 응답
print(result.last_agent.name)
print(result.final_output)

Billing_agent
청구서를 확인해드릴게요. 오류로 보이는 부분을 알려주시면 바로 도와드리겠습니다.  
예: 중복 청구, 금액 오류, 사용 내역 불일치, 환불 누락 등


## 2. on_handoff 콜백

`on_handoff` 콜백은 핸드오프가 호출되는 순간 실행됩니다.  
데이터 준비, 로깅, 알림 등 사이드 이펙트 처리에 유용합니다.

- `input_type` 없이 사용 시: `def on_handoff(ctx: RunContextWrapper[None])`  
- `input_type` 함께 사용 시: `async def on_handoff(ctx, input_data: MyModel)` (섹션 3 참고)

In [4]:
from agents import Agent, Runner, handoff, RunContextWrapper

# --- 1. 핸드오프 콜백(callback) 함수 정의 ---

# 핸드오프가 실행되는 시점에 자동 호출되는 함수
def on_billing_handoff(ctx: RunContextWrapper[None]):
    """Billing Agent로 핸드오프될 때 실행되는 콜백"""
    print("[로그] Billing Agent로 핸드오프 발생")

def on_refund_handoff(ctx: RunContextWrapper[None]):
    """Refund Agent로 핸드오프될 때 실행되는 콜백"""
    print("[로그] Refund Agent로 핸드오프 발생")

# --- 2. 하위 에이전트 정의 ---

# 청구 전담 에이전트
billing_agent = Agent(
    name="Billing_agent",
    instructions="당신은 청구 관련 질문을 전문으로 처리합니다.",
    model=Model
)

# 환불 전담 에이전트
refund_agent = Agent(
    name="Refund_agent",
    instructions="당신은 환불 요청을 전문으로 처리합니다.",
    model=Model
)

# --- 3. 트리아지(분류) 에이전트 정의 ---
triage_agent = Agent(
    name="Triage_agent",
    instructions="사용자 요청을 분류하여 적합한 에이전트에게 넘겨주세요.",
    model=Model,
    handoffs=[
        # on_handoff 파라미터로 콜백 등록
        # 핸드오프 실행 직전에 해당 콜백이 호출됨
        handoff(billing_agent, on_handoff=on_billing_handoff),
        handoff(refund_agent, on_handoff=on_refund_handoff),
    ],
)

# --- 4. 실행 ---
# 입력: "환불을 받고 싶습니다." → triage_agent가 환불 요청으로 분류
# → on_refund_handoff 콜백 실행 → refund_agent로 핸드오프 → 최종 응답 생성
result = await Runner.run(triage_agent, input="환불을 받고 싶습니다.")

print()
print(result.last_agent.name)
print(result.final_output)

[로그] Refund Agent로 핸드오프 발생

Refund_agent
환불 요청으로 연결했습니다.  
환불 사유와 주문 정보를 알려주시면 바로 도와드리겠습니다.


## 3. Handoff 입력 데이터 (input_type)

`input_type`을 사용하면 LLM이 핸드오프 시 **구조화된 데이터를 함께 전달**하도록 할 수 있습니다.  
예를 들어, 에스컬레이션 이유를 함께 전달받아 로깅하거나 처리에 활용할 수 있습니다.

- `input_type`이 있을 때 `on_handoff`는 `async def`로 정의하고 두 번째 인자로 `input_data`를 받습니다.

In [5]:
from pydantic import BaseModel
from agents import Agent, Runner, handoff, RunContextWrapper
from datetime import datetime

# 현재 시간을 문자열로 생성 (에이전트 프롬프트에 사용)
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 에스컬레이션 시 전달할 데이터 구조 정의
# Pydantic 모델을 사용하면 에이전트가 전달해야 하는 입력 형식을 강제할 수 있음
class EscalationData(BaseModel):
    reason: str          # 에스컬레이션 사유
    datetime: str        # 접수 시간


# 에스컬레이션이 발생했을 때 실행되는 콜백 함수
# ctx : 실행 컨텍스트
# input_data : 에이전트가 전달한 EscalationData 객체
async def on_escalation(ctx: RunContextWrapper[None], input_data: EscalationData):
    print(f"[에스컬레이션] 이유: {input_data.reason}, 접수시간: {input_data.datetime}")


# 복잡한 문제를 처리하는 전문 상담 에이전트
# Support agent가 처리하지 못하는 문제를 넘겨받아 해결
escalation_agent = Agent(
    name="Escalation_agent",
    instructions="당신은 복잡한 문제를 처리하는 전문 상담사입니다.",
    model=Model
)

# 일반 고객 지원을 처리하는 1차 상담 에이전트
support_agent = Agent(
    name="Support_agent",
    instructions=(
        f"현재 시간은 {now} 입니다."
        "일반 고객 지원 요청을 처리하세요. "
        "복잡하거나 민감한 문제는 에스컬레이션 에이전트에게 이유와 접수시간을 함께 넘겨주세요."
    ),
    model=Model,

    # 다른 에이전트로 작업을 넘기는 handoff 설정
    handoffs=[
        handoff(
            escalation_agent,      # 넘겨받을 대상 에이전트
            on_handoff=on_escalation,  # handoff 발생 시 실행할 콜백
            input_type=EscalationData, # 전달해야 하는 데이터 형식
        ),
    ],
)

# Support agent 실행
# 사용자의 입력을 전달하면 필요 시 escalation_agent로 handoff 발생
result = await Runner.run(
    support_agent,
    input="제 계정이 해킹당한 것 같습니다. 즉시 도움이 필요합니다!"
)

# 최종 에이전트 응답 출력
print()
print(result.last_agent.name)
print(result.final_output)

[에스컬레이션] 이유: 사용자가 계정 해킹 의심으로 즉시 도움이 필요하다고 요청함. 보안/계정 침해 가능성이 있는 민감한 사안이므로 즉시 에스컬레이션 필요., 접수시간: 2026-07-03 13:16:47

Escalation_agent
즉시 조치하세요:

1. **비밀번호 변경**: 해당 계정과 연결된 이메일부터 먼저 변경
2. **모든 기기 로그아웃**
3. **2단계 인증(2FA) 켜기**
4. **복구 이메일/전화번호 확인**
5. **최근 로그인/결제 내역 확인**
6. **의심스러운 앱/연동 해제**
7. **같은 비밀번호를 쓴 다른 계정도 변경**

가능하면 지금 바로:
- 계정 서비스명
- 마지막으로 정상 접속한 시각
- 이상 징후(로그인 알림, 메일 변경, 결제 등)

를 알려주시면 다음 조치를 바로 안내하겠습니다.


## 4. Input Filter (입력 필터)

핸드오프가 발생하면 새로운 에이전트는 기존 대화 기록 전체를 받습니다.  
`input_filter`를 사용하면 다음 에이전트에게 전달되는 **대화 기록을 가공**할 수 있습니다.

`agents.extensions.handoff_filters`에는 자주 사용되는 내장 필터가 제공됩니다:

| 필터 | 설명 |
|------|------|
| `remove_all_tools` | 대화 기록에서 모든 도구 호출/결과 제거 |
|`nest_handoff_history` | 이전 대화 내용을 요약해서 단일 assistant 메시지로 압축 후 전달 |

### 흐름 설명
```
사용자: "주문번호 ORD-1234 배송 조회해주고, 반품 정책도 알려줘"
      ↓
main_agent
  → check_shipping_status("ORD-1234") 도구 호출  ← 도구 메시지 생성
  → "반품 정책은 FAQ agent한테 넘겨야겠다"
  → remove_all_tools 필터 실행
      ┌─────────────────────────────────────┐
      │ 도구 호출 기록 제거 전                 │
      │ [user] 배송 조회해주고 반품도 알려줘    │
      │ [tool_call] check_shipping_status   │  ← 제거됨
      │ [tool_result] 배송 중...             │  ← 제거됨
      └─────────────────────────────────────┘
      ↓
faq_agent (도구 기록 없이 깔끔한 대화만 전달)
  → 반품 정책 답변
```

In [6]:
from agents import Agent, Runner, handoff, function_tool
from agents.extensions import handoff_filters

# -------------------------------
# 도구(tool) 정의
# -------------------------------

@function_tool
def check_shipping_status(order_id: str) -> str:
    """
    주문 ID로 배송 상태를 조회하는 도구 함수
    function_tool 데코레이터를 사용하면
    LLM이 호출 가능한 Tool로 자동 등록됨
    """
    print(f"** check_shipping_status 실행 ** order_id: {order_id}")

    # 실제 시스템에서는 DB 또는 API 조회
    return f"주문 {order_id}의 배송 상태: 배송 중 (예상 도착: 내일)"


# -------------------------------
# FAQ 에이전트 정의
# -------------------------------
faq_agent = Agent(
    name="FAQ_agent",
    # FAQ 관련 질문을 담당하는 에이전트
    instructions="자주 묻는 질문에 답변합니다. 대화 기록에 도구 호출 내역이 있는지 확인하세요.",

    model=Model
)


# -------------------------------
# 메인 에이전트 정의
# -------------------------------
main_agent = Agent(
    name="Main_agent",
    # 역할 정의
    instructions=(
        "당신은 배송 조회만 담당합니다. "
        "배송 조회는 check_shipping_status 도구를 사용하세요. "
        "반품, 환불, 정책 등 FAQ 성격의 질문에는 직접 답변하지 말고 "
        "반드시 FAQ agent에게 핸드오프하세요."
    ),

    model=Model,
    # 이 에이전트가 사용할 수 있는 Tool 목록
    tools=[check_shipping_status],

    # 다른 에이전트에게 작업을 넘기는 handoff 설정
    handoffs=[
        handoff(
            faq_agent,  # 작업을 넘길 대상 에이전트

            # handoff 시 tool 호출 기록 등을 제거하는 필터
            # (주석 해제하면 tool call 기록을 제거하고 전달)
            # input_filter=handoff_filters.remove_all_tools,
        ),
    ],
)


# -------------------------------
# 에이전트 실행
# -------------------------------
result = await Runner.run(
    main_agent,

    # 사용자 입력
    # 배송조회 + FAQ 질문이 동시에 포함된 요청
    input="주문번호 ORD-1234 배송 조회해주고, 반품 정책도 알려줘"
)

# 최종 에이전트 응답 출력
print(f"\n최종 응답 에이전트: {result.last_agent.name}")
print(result.final_output)

** check_shipping_status 실행 ** order_id: ORD-1234

최종 응답 에이전트: FAQ_agent
주문번호 **ORD-1234**는 현재 **배송 중**이며, **예상 도착은 내일**입니다.

반품 정책은 **FAQ 담당자에게 연결해 두었습니다**. 원하시면 제가 이어서 반품 정책 요약도 정리해드릴게요.


In [7]:
# 지금까지의 대화 기록을 다음 Runner.run 호출에서
# 그대로 이어서 사용할 수 있는 형태로 만들어 줌
result.to_input_list()

[{'content': '주문번호 ORD-1234 배송 조회해주고, 반품 정책도 알려줘', 'role': 'user'},
 {'arguments': '{"order_id":"ORD-1234"}',
  'call_id': 'call_6JCffjwsgbPxnfdUGLsSZp4j',
  'name': 'check_shipping_status',
  'type': 'function_call',
  'id': 'fc_00462201c6f51a11006a4737b3ca808198a15ba91e8f64c369',
  'status': 'completed'},
 {'call_id': 'call_6JCffjwsgbPxnfdUGLsSZp4j',
  'output': '주문 ORD-1234의 배송 상태: 배송 중 (예상 도착: 내일)',
  'type': 'function_call_output'},
 {'arguments': '{}',
  'call_id': 'call_fYfMVSUEz7f1cBnAHNJTqwRc',
  'name': 'transfer_to_faq_agent',
  'type': 'function_call',
  'id': 'fc_00462201c6f51a11006a4737b500c481989b44242dd8cdc8c4',
  'status': 'completed'},
 {'call_id': 'call_fYfMVSUEz7f1cBnAHNJTqwRc',
  'output': '{"assistant": "FAQ_agent"}',
  'type': 'function_call_output'},
 {'id': 'msg_00462201c6f51a11006a4737b5a008819885601edb5c0ed3d2',
  'content': [{'annotations': [],
    'text': '주문번호 **ORD-1234**는 현재 **배송 중**이며, **예상 도착은 내일**입니다.\n\n반품 정책은 **FAQ 담당자에게 연결해 두었습니다**. 원하시면 제가 이어서 반품 

## 5. 권장 프롬프트 (RECOMMENDED_PROMPT_PREFIX)

LLM이 핸드오프를 올바르게 이해하고 활용하려면 관련 정보를 프롬프트에 포함시키는 것이 좋습니다.  
```
LLM은 핸드오프가 뭔지 기본적으로 모릅니다. 그래서 "너는 다른 에이전트에게 작업을 넘길 수 있어" 라고 미리 알려줘야 합니다.

핸드오프 설명 없이:
  LLM: "나는 그냥 질문에 답하는 AI야"
  → 핸드오프 도구가 있어도 잘 안 씀

핸드오프 설명 포함:
  LLM: "나는 다른 전문 에이전트에게 작업을 위임할 수 있어"
  → 핸드오프를 적극적으로 활용
```
`agents.extensions.handoff_prompt`에서 권장 프롬프트를 제공합니다:

| 방법 | 설명 |
|------|------|
| `RECOMMENDED_PROMPT_PREFIX` | 프롬프트 앞에 직접 붙여 사용하는 문자열 상수 |
| `prompt_with_handoff_instructions(prompt)` | 기존 프롬프트에 권장 내용을 자동으로 추가하는 함수 |

```
# 시스템 컨텍스트
당신은 Agents SDK라는 멀티 에이전트 시스템의 일부입니다.
이 시스템은 에이전트 간의 조율과 실행을 쉽게 만들기 위해 설계되었습니다.

Agents SDK는 두 가지 핵심 추상화를 사용합니다: **에이전트(Agents)** 와 **핸드오프(Handoffs)**.

에이전트는 지시사항(instructions)과 도구(tools)를 포함하며,
필요한 경우 대화를 다른 에이전트에게 넘길 수 있습니다.

핸드오프는 일반적으로 `transfer_to_<에이전트명>` 형태로 명명된
핸드오프 함수를 호출함으로써 이루어집니다.

에이전트 간의 전환은 백그라운드에서 원활하게 처리되므로,
사용자와의 대화에서 이러한 전환을 언급하거나 드러내지 마세요.
```

In [8]:
from agents import Agent, Runner, handoff
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX, prompt_with_handoff_instructions

RECOMMENDED_PROMPT_PREFIX

'# System context\nYou are part of a multi-agent system called the Agents SDK, designed to make agent coordination and execution easy. Agents uses two primary abstraction: **Agents** and **Handoffs**. An agent encompasses instructions and tools and can hand off a conversation to another agent when appropriate. Handoffs are achieved by calling a handoff function, generally named `transfer_to_<agent_name>`. Transfers between agents are handled seamlessly in the background; do not mention or draw attention to these transfers in your conversation with the user.\n'

In [9]:
prompt_with_handoff_instructions("당신은 환불 처리 전문가입니다.")

'# System context\nYou are part of a multi-agent system called the Agents SDK, designed to make agent coordination and execution easy. Agents uses two primary abstraction: **Agents** and **Handoffs**. An agent encompasses instructions and tools and can hand off a conversation to another agent when appropriate. Handoffs are achieved by calling a handoff function, generally named `transfer_to_<agent_name>`. Transfers between agents are handled seamlessly in the background; do not mention or draw attention to these transfers in your conversation with the user.\n\n\n당신은 환불 처리 전문가입니다.'

In [10]:
# 청구/결제 전문 에이전트
billing_agent = Agent(
    name="Billing_agent",
    instructions=f"""{RECOMMENDED_PROMPT_PREFIX}
당신은 청구 및 결제 관련 질문을 처리하는 전문가입니다.""",
    model=Model
)

# 환불 전문 에이전트
refund_agent = Agent(
    name="Refund_agent",
    instructions=prompt_with_handoff_instructions("당신은 환불 처리 전문가입니다."),
    model=Model
)

# 사용자 요청을 분석하여 적절한 전문 에이전트로 라우팅하는 triage 에이전트
triage_agent = Agent(
    name="Triage_agent",
    instructions=prompt_with_handoff_instructions(
        "사용자 요청을 분석하여 청구 질문은 Billing agent에게, 환불 요청은 Refund agent에게 넘겨주세요."
    ),
    model=Model,
    handoffs=[billing_agent, refund_agent],  # 핸드오프 가능한 에이전트 목록
)

# triage_agent 실행 → 청구 관련 질문이므로 billing_agent로 핸드오프 예상
result = await Runner.run(triage_agent, input="지난달 청구 금액이 잘못된 것 같습니다.")
print(result.final_output)

지난달 청구 금액에 문제가 있는 것 같군요. 확인을 도와드리겠습니다.

가능하면 아래 정보를 알려주세요:
- 청구서 번호
- 청구된 금액
- 예상하신 금액
- 해당 청구가 발생한 날짜 또는 서비스명

주시면 바로 확인해볼게요.


## 6. 종합 예제 - 고객 지원 시스템

여러 핸드오프 기능을 조합한 고객 지원 시스템입니다.

```
사용자
  └─→ Triage Agent (분류)
        ├─→ Order Agent    (주문 조회)
        ├─→ Refund Agent   (환불 처리, on_handoff + input_type)
        └─→ FAQ Agent      (일반 문의, input_filter 적용)
```

In [11]:
from pydantic import BaseModel
from agents import Agent, Runner, handoff, RunContextWrapper  #에이전트 실행 중 공유 상태(Context)에 접근하는 래퍼(wrapper)
from agents.extensions import handoff_filters
from agents.extensions.handoff_prompt import prompt_with_handoff_instructions

# 환불 요청 데이터 구조 정의 (LLM이 핸드오프 시 채워서 전달)
class RefundRequest(BaseModel):
    order_id: str  # 주문번호
    reason: str    # 환불 사유

# 환불 핸드오프 발생 시 실행되는 콜백 함수 (로깅 용도)
async def on_refund_handoff(ctx: RunContextWrapper[None], input_data: RefundRequest):
    print(f"[환불 요청 접수] 주문번호: {input_data.order_id}, 사유: {input_data.reason}")

# 주문/배송 전문 에이전트
order_agent = Agent(
    name="Order_Agent",
    instructions=prompt_with_handoff_instructions(
        "주문 상태 및 배송 관련 문의를 처리합니다. 주문번호를 확인하고 현황을 안내하세요."
    ),
    model=Model
)

# 환불 전문 에이전트
refund_agent = Agent(
    name="Refund_Agent",
    instructions=prompt_with_handoff_instructions(
        "환불 요청을 처리합니다. 고객에게 환불 절차와 소요 시간을 안내하세요."
    ),
    model=Model
)

# FAQ 전문 에이전트
faq_agent = Agent(
    name="FAQ_Agent",
    instructions=prompt_with_handoff_instructions(
        "자주 묻는 질문에 답변합니다. 배송, 반품 정책, 결제 방법 등 일반적인 문의를 처리하세요."
    ),
    model=Model
)

# 고객 문의를 분류하여 적절한 전문 에이전트로 라우팅하는 triage 에이전트
triage_agent = Agent(
    name="Triage_Agent",
    instructions=prompt_with_handoff_instructions(
        "고객 문의를 분류하세요:\n"
        "- 주문/배송 조회 → Order Agent\n"
        "- 환불 요청 → Refund Agent (주문번호와 사유 필요)\n"
        "- 일반 문의/FAQ → FAQ Agent"
    ),
    model=Model,
    handoffs=[
        order_agent,   # 주문 에이전트: 기본 핸드오프 (옵션 없음)
        handoff(
            refund_agent,   # 환불 에이전트: 핸드오프 시 on_refund_handoff 콜백 실행
            on_handoff=on_refund_handoff,  # 핸드오프 발생 시 로깅
            input_type=RefundRequest,       # LLM이 채워야 할 구조화된 입력
        ),
        handoff(
            faq_agent,   # FAQ 에이전트: 도구 호출 기록을 제거하고 깔끔한 대화만 전달
            input_filter=handoff_filters.remove_all_tools,  # 도구 메시지 제거
        ),
    ],
)

In [13]:
print("=== 테스트 1: 주문 조회 ===")
result = await Runner.run(triage_agent, "주문번호 ORD-1234의 배송 현황을 알고 싶습니다.")

print(result.last_agent.name)
print(result.final_output)

=== 테스트 1: 주문 조회 ===
Order_Agent
주문번호 ORD-1234의 배송 현황을 확인해드리겠습니다. 잠시만 기다려 주세요.


In [14]:
print("=== 테스트 2: 환불 요청 ===")
result = await Runner.run(triage_agent, "주문번호 ORD-5678 상품을 환불하고 싶습니다. 사이즈가 맞지 않아서요.")

print(result.last_agent.name)
print(result.final_output)

=== 테스트 2: 환불 요청 ===
[환불 요청 접수] 주문번호: ORD-5678, 사유: 사이즈가 맞지 않아서
Refund_Agent
환불 도와드리겠습니다.

주문번호 **ORD-5678** 확인했습니다.  
사이즈가 맞지 않아 환불을 원하시는 경우, 아래 절차로 진행됩니다.

1. **환불 접수**  
   주문 정보를 바탕으로 환불 요청이 등록됩니다.

2. **상품 회수/반송 확인**  
   상품 상태와 반송 여부를 확인합니다.

3. **환불 승인 및 처리**  
   확인이 완료되면 결제 수단으로 환불이 진행됩니다.

**소요 시간**  
- 접수 후 확인까지: 보통 **1~2영업일**  
- 환불 완료까지: 반송/검수 후 **3~7영업일** 정도 소요될 수 있습니다.

원하시면 제가 바로 **환불 진행**을 이어서 안내해드리겠습니다.


In [15]:
print("=== 테스트 3: 일반 문의 ===")
result = await Runner.run(triage_agent, "반품 정책이 어떻게 되나요?")

print(result.last_agent.name)
print(result.final_output)

=== 테스트 3: 일반 문의 ===
FAQ_Agent
반품 정책은 상품 수령 후 **7일 이내**에 신청 가능합니다.

- **반품 가능 조건:** 미사용, 원래 포장 상태 유지
- **반품 불가:** 사용 흔적이 있거나 훼손된 경우
- **비용:** 단순 변심은 고객 부담, 상품 하자는 당사 부담

원하시면 **교환/환불 절차**도 바로 안내해드릴게요.


### 실습 문제

아래 요구사항에 맞는 **여행 예약 지원 시스템**을 구현하세요.

**에이전트 구성:**

1. **Triage Agent**: 사용자 요청을 분류하여 적절한 에이전트에 핸드오프
2. **Flight Agent**: 항공편 예약 및 조회 처리
3. **Hotel Agent**: 호텔 예약 및 조회 처리
4. **Cancellation Agent**: 예약 취소 처리 (`on_handoff` 콜백으로 취소 정보 로깅)

**요구사항:**
- `Cancellation Agent`로 핸드오프 시 `input_type`으로 예약 번호(`booking_id: str`)와 취소 사유(`reason: str`)를 전달받아 출력
- 모든 에이전트에 `prompt_with_handoff_instructions` 적용
- `Flight Agent`, `Hotel Agent`는 `handoff()` 함수 형태로 지정

**테스트 입력:**
- `"서울-제주 항공편을 예약하고 싶습니다."` → Flight Agent 응답
- `"제주도 호텔을 3박 예약하려고 합니다."` → Hotel Agent 응답
- `"예약번호 BK-999 항공편을 취소하고 싶어요. 일정이 바뀌어서요."` → Cancellation Agent 핸드오프 + 콜백 출력

In [21]:
from pydantic import BaseModel
from agents import Agent, Runner, handoff, RunContextWrapper
from agents.extensions.handoff_prompt import prompt_with_handoff_instructions

# 취소 요청 시 전달할 데이터 구조 (LLM이 채워서 전달)
class CancellationRequest(BaseModel):
    booking_id: str  # 예약 번호
    reason: str      # 취소 사유

# Cancellation Agent로 핸드오프될 때 실행되는 콜백 (취소 정보 로깅)
async def on_cancellation(ctx: RunContextWrapper[None], input_data: CancellationRequest):
    print(f"[취소 요청 접수] 예약번호: {input_data.booking_id}, 사유: {input_data.reason}")

In [22]:
# 항공편 예약 및 조회 전문 에이전트
flight_agent = Agent(
    name="Flight_Agent",
    instructions=prompt_with_handoff_instructions(
        "항공편 예약 및 조회를 처리합니다. 출발지, 도착지, 날짜를 확인하고 예약을 안내하세요."
    ),
    model=Model,
)

# 호텔 예약 및 조회 전문 에이전트
hotel_agent = Agent(
    name="Hotel_Agent",
    instructions=prompt_with_handoff_instructions(
        "호텔 예약 및 조회를 처리합니다. 지역, 숙박 일수, 인원을 확인하고 예약을 안내하세요."
    ),
    model=Model,
)

# 예약 취소 전문 에이전트
cancellation_agent = Agent(
    name="Cancellation_Agent",
    instructions=prompt_with_handoff_instructions(
        "예약 취소 요청을 처리합니다. 고객에게 취소 절차와 환불 규정을 안내하세요."
    ),
    model=Model,
)

In [23]:
triage_agent = Agent(
    name="Triage_Agent",
    instructions=prompt_with_handoff_instructions(
        "여행 예약 관련 고객 요청을 분류하세요:\n"
        "- 항공편 예약/조회 → Flight Agent\n"
        "- 호텔 예약/조회 → Hotel Agent\n"
        "- 예약 취소 → Cancellation Agent (예약번호와 취소 사유 필요)"
    ),
    model=Model,
    handoffs=[
        # handoff() 함수 형태로 지정 (요구사항)
        handoff(flight_agent),
        handoff(hotel_agent),
        handoff(
            cancellation_agent,
            on_handoff=on_cancellation,        # 핸드오프 발생 시 취소 정보 로깅
            input_type=CancellationRequest,    # LLM이 채워야 할 구조화된 입력
        ),
    ],
)

In [29]:
print("=== 테스트 1: 항공편 예약 ===")
result = await Runner.run(triage_agent, "서울-제주 항공편을 예약하고 싶습니다.")

print(result.last_agent.name)
print(result.final_output)

=== 테스트 1: 항공편 예약 ===
Flight_Agent
예약을 도와드릴게요.  
항공편 예약을 위해 아래 정보가 필요합니다:

- 출발 날짜
- 탑승 인원
- 원하는 시간대(선택)
- 편도/왕복 여부

예시: “3월 15일, 2명, 오전, 편도”


In [25]:
print("=== 테스트 2: 호텔 예약 ===")
result = await Runner.run(triage_agent, "제주도 호텔을 3박 예약하려고 합니다.")

print(result.last_agent.name)
print(result.final_output)

=== 테스트 2: 호텔 예약 ===
Hotel_Agent
제주도 호텔 3박 예약 도와드릴게요.  
예약을 위해 아래 정보를 알려주세요.

1. 숙박 인원  
2. 체크인 날짜  
3. 객실 타입 선호(예: 더블, 트윈, 스위트)  
4. 예산 범위(선택)


OPENAI_API_KEY is not set, skipping trace export


In [19]:
print("=== 테스트 3: 예약 취소 ===")
result = await Runner.run(triage_agent, "예약번호 BK-999 항공편을 취소하고 싶어요. 일정이 바뀌어서요.")

print(result.last_agent.name)
print(result.final_output)

=== 테스트 3: 예약 취소 ===
[취소 요청 접수] 예약번호: BK-999, 사유: 일정이 바뀌어서요.


OPENAI_API_KEY is not set, skipping trace export


Cancellation_Agent
예약 취소를 도와드릴게요.

취소는 가능하며, 항공권 종류와 출발일까지 남은 시간에 따라 환불 규정이 달라집니다.  
일반적으로:
- **출발 전일수록** 수수료가 낮을 수 있음
- **특가/비환불 운임**은 환불이 제한될 수 있음
- 환불은 보통 **결제 수단으로 3~10영업일** 내 처리됩니다

예약번호 **BK-999**로 취소 요청을 접수했습니다.  
원하시면 제가 이어서 **취소 가능 여부와 환불 예상 금액**까지 확인해드릴게요.


OPENAI_API_KEY is not set, skipping trace export
